In [1]:
"""
EXPERIMENT 5: Monte Carlo Prediction & Control
Key Idea: Learn from COMPLETE episodes. No bootstrapping.
"""
import numpy as np
from collections import defaultdict

# --- Grid World: 4x4, Goal=(3,3), Holes=(1,1),(2,2) ---
class GridWorld:
    def reset(self): self.s = (0,0); return self.s
    def step(self, a):
        r,c = self.s
        if a==0: r=max(r-1,0)
        elif a==1: r=min(r+1,3)
        elif a==2: c=max(c-1,0)
        else: c=min(c+1,3)
        self.s = (r,c)
        if self.s==(3,3): return self.s,+10,True   # goal
        if self.s in [(1,1),(2,2)]: return self.s,-10,True  # hole
        return self.s,-1,False

env = GridWorld()

# ── PART A: MC Prediction ──────────────────────────────
# Estimate V(s) by averaging returns from complete episodes
V = defaultdict(float)
returns = defaultdict(list)

for _ in range(500):
    traj, s, done = [], env.reset(), False
    while not done:
        a = np.random.randint(4)                  # random policy
        ns, r, done = env.step(a)
        traj.append((s, r)); s = ns

    G, seen = 0, set()
    for s, r in reversed(traj):
        G = r + 0.9 * G                           # discounted return
        if s not in seen:                         # first-visit only
            seen.add(s); returns[s].append(G)
            V[s] = np.mean(returns[s])            # average all G's seen

print("── MC Prediction: V(s) ──")
for r in range(4): print([round(V[(r,c)],1) for c in range(4)])

# ── PART B: MC Control (e-greedy) ─────────────────────
# Learn optimal policy via Q(s,a) - update after full episodes
Q = defaultdict(lambda: np.zeros(4))
qreturns = defaultdict(list)

for ep in range(3000):
    traj, s, done = [], env.reset(), False
    while not done:
        a = np.random.randint(4) if np.random.rand()<0.1 else np.argmax(Q[s])
        ns, r, done = env.step(a)
        traj.append((s,a,r)); s = ns

    G, seen = 0, set()
    for s,a,r in reversed(traj):
        G = r + 0.9*G
        if (s,a) not in seen:
            seen.add((s,a)); qreturns[(s,a)].append(G)
            Q[s][a] = np.mean(qreturns[(s,a)])

arrows = ['^','v','<','>']
print("\n── MC Control: Policy ──")
for r in range(4): print([arrows[np.argmax(Q[(r,c)])] for c in range(4)])

── MC Prediction: V(s) ──
[np.float64(-9.7), np.float64(-9.7), np.float64(-9.0), np.float64(-7.9)]
[np.float64(-9.7), 0.0, np.float64(-9.0), np.float64(-6.6)]
[np.float64(-9.2), np.float64(-9.4), 0.0, np.float64(0.0)]
[np.float64(-8.0), np.float64(-7.0), np.float64(-3.1), 0.0]

── MC Control: Policy ──
['>', '>', '>', 'v']
['^', '^', '>', 'v']
['^', '<', '^', 'v']
['>', '<', '^', '^']
